In [0]:
%sql
use catalog projectcatalog;


In [0]:
%sql
use SliverSchemaSales

In [0]:
%sql
select * from sparkschemaprojectsales.tblprjregionsource

region_id,region
R01,East
R02,West
R03,North
R04,South


In [0]:
from pyspark.sql.functions import *
bronze_df = spark.read.table("sparkschemaprojectsales.tblprjregionsource")
display(bronze_df)

region_id,region
R01,East
R02,West
R03,North
R04,South


In [0]:
from pyspark.sql.functions import *
bronze_df = spark.read.table("sparkschemaprojectsales.tblprjregionsource")

silver_df = (
    bronze_df
   
    # Standardization
    .withColumn("region_id", upper(col("region_id")))
    .withColumn("region", initcap(col("region")))
    # Trim spaces # replacing blank values with null
     .withColumn(
        "region_id",
        when(trim(col("region_id")) == "", None)
        .otherwise(trim(col("region_id")))
    )
    .withColumn(
        "region",
        when(trim(col("region")) == "", None)
        .otherwise(trim(col("region")))
    )
  
    # checking null values
    .filter(col("region_id").isNotNull())
    .filter(col("region").isNotNull())    
    # Validate Product ID Format
    .filter(col("region_id").rlike(r"^R[0-9]{2}$") )
    # Remove duplicate products
    .dropDuplicates(["region_id"])
    # Check and remove Duplicate Product Names
    .dropDuplicates(["region"])
    #  Add Audit Columns    
    .withColumn( "silver_load_created_date", current_timestamp() ) 
    .withColumn( "source_file", lit("RegionsFile.parquet") ) 
    .withColumn( "data_layer", lit("SILVER") )
    .orderBy( col("region_id").asc() )
)

display(silver_df)

region_id,region,silver_load_created_date,source_file,data_layer
R01,East,2026-08-11T18:32:01.306Z,RegionsFile.parquet,SILVER
R02,West,2026-08-11T18:32:01.306Z,RegionsFile.parquet,SILVER
R03,North,2026-08-11T18:32:01.306Z,RegionsFile.parquet,SILVER
R04,South,2026-08-11T18:32:01.306Z,RegionsFile.parquet,SILVER


In [0]:
silver_df.write.mode("overwrite").saveAsTable("tblprjregionssilver")

In [0]:
silver_check_df = spark.table("SliverSchemaSales.tblprjregionssilver")
display(silver_check_df)

region_id,region,silver_load_created_date,source_file,data_layer
R01,East,2026-08-11T18:32:08.835Z,RegionsFile.parquet,SILVER
R02,West,2026-08-11T18:32:08.835Z,RegionsFile.parquet,SILVER
R03,North,2026-08-11T18:32:08.835Z,RegionsFile.parquet,SILVER
R04,South,2026-08-11T18:32:08.835Z,RegionsFile.parquet,SILVER


In [0]:
%sql
select count(*) as bronzedatacnt from sparkschemaprojectsales.tblprjregionsource


bronzedatacnt
4


In [0]:
%sql
select count(*) as silverdatacnt from tblprjregionssilver

silverdatacnt
4
